# จากบทเรียนสู่โปรเจกต์ CNN

Notebook นี้เชื่อมเนื้อหา Week 2–7 กับโปรเจกต์จำแนกตัวอักษรไทย โดยใช้ตัวอย่างเล็ก ๆ ที่รันได้บน CPU

เมื่อเรียนจบควรตอบได้ว่า:

1. ภาพกลายเป็นตัวเลขที่โมเดลอ่านได้อย่างไร
2. convolution, padding, activation และ pooling ทำอะไร
3. loss, backpropagation และ optimizer ทำให้โมเดลเรียนได้อย่างไร
4. CNN ในโปรเจกต์ประกอบด้วยอะไร
5. augmentation, transfer learning และ metrics ช่วยการทดลองอย่างไร

Week 1 กล่าวถึงข้อมูลสุขภาพ จึงไม่ได้ใช้โดยตรงกับชุดข้อมูลตัวอักษรไทย แต่หลักการตรวจคุณภาพและรักษาความหมายของข้อมูลยังใช้เหมือนกัน


## เตรียม notebook

เลือก kernel `.venv` ของโปรเจกต์ แล้วรันทุก cell ตามลำดับ Notebook นี้ไม่ใช้ dataset จริงและไม่ดาวน์โหลด pretrained weights


In [ ]:
from pathlib import Path
import sys

project_dir = next(
    (folder.resolve() for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "src/train.py").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("เปิด notebook จากภายในโฟลเดอร์โปรเจกต์")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch import nn

from src.train import TrainConfig, build_model, make_transform

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Project:", project_dir)


## 1. ภาพคือ array ของตัวเลข — Week 2–3

ภาพ grayscale มีรูปทรง `(สูง, กว้าง)` ส่วนภาพ RGB มี `(สูง, กว้าง, 3)` โดย 3 ช่องแทนสีแดง เขียว และน้ำเงิน

ตัวอย่างด้านล่างสร้างภาพคล้ายตัวอักษรจาก NumPy ค่า `0` คือสีดำ และ `1` คือสีขาว


In [ ]:
gray_image = np.ones((12, 12), dtype=np.float32)
gray_image[2:10, 5:7] = 0
gray_image[2:4, 3:9] = 0
gray_image[6:8, 3:9] = 0

rgb_image = np.repeat(gray_image[..., None], 3, axis=2)

print("Grayscale shape:", gray_image.shape)
print("RGB shape:", rgb_image.shape)
print("Pixel ตรงกลาง:", gray_image[6, 6])

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(gray_image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Grayscale")
axes[1].imshow(rgb_image)
axes[1].set_title("RGB")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()


### RGB กับ HSV ใช้มองสีคนละแบบ

RGB เก็บความเข้มของแสงสีแดง เขียว และน้ำเงิน ส่วน HSV แยกชนิดสี ความอิ่มสี และความสว่าง งานแยกวัตถุตามสีมักใช้ HSV ได้ง่ายกว่า

โปรเจกต์นี้ใช้ RGB เพราะ backbone pretrained เรียนด้วย RGB และตัวอักษรสนใจรูปร่างมากกว่าสี จึงยังไม่ต้องแปลงเป็น HSV


In [ ]:
from matplotlib.colors import rgb_to_hsv

color_names = ["red", "green", "blue"]
rgb_samples = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=np.float32)
hsv_samples = rgb_to_hsv(rgb_samples)

display(pd.DataFrame({
    "color": color_names,
    "RGB": [row.round(2).tolist() for row in rgb_samples],
    "HSV": [row.round(2).tolist() for row in hsv_samples],
}))


### Histogram บอกการกระจายความเข้มของภาพ

Histogram นับว่ามีพิกเซลมืดหรือสว่างเท่าไร ใช้ตรวจพื้นหลัง แสง และ contrast ได้ แต่ไม่บอกตำแหน่งของพิกเซล จึงแยกตัวอักษรที่มีจำนวนพิกเซลใกล้กันได้ไม่ดี


In [ ]:
counts, edges = np.histogram(gray_image, bins=4, range=(0, 1))
plt.bar(edges[:-1], counts, width=np.diff(edges), align="edge", edgecolor="black")
plt.xlabel("Pixel intensity")
plt.ylabel("Number of pixels")
plt.title("Grayscale histogram")
plt.show()


## 2. Convolution และ kernel — Week 4

Kernel เลื่อนผ่านภาพและคำนวณผลรวมของ `บริเวณภาพ × ค่าน้ำหนัก` แต่ละ kernel จึงตอบสนองต่อลักษณะต่างกัน เช่น เส้นตั้ง เส้นนอน หรือขอบ

ใน PyTorch `Conv2d` คำนวณ cross-correlation และเรียนค่าน้ำหนักของ kernel จากข้อมูล ต่างจากตัวกรองภาพแบบดั้งเดิมที่เรากำหนด kernel เอง


In [ ]:
image_tensor = torch.from_numpy(gray_image)[None, None]
vertical_edge_kernel = torch.tensor(
    [[[-1.0, 0.0, 1.0],
      [-1.0, 0.0, 1.0],
      [-1.0, 0.0, 1.0]]]
)[None]

edge_map = F.conv2d(image_tensor, vertical_edge_kernel, padding=1)

patch = image_tensor[0, 0, 1:4, 2:5]
first_value = (patch * vertical_edge_kernel[0, 0]).sum()
print("ตัวอย่าง patch:\n", patch)
print("ผลรวม patch × kernel:", first_value.item())

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(gray_image, cmap="gray")
axes[0].set_title("Input")
axes[1].imshow(vertical_edge_kernel[0, 0], cmap="coolwarm")
axes[1].set_title("Kernel")
axes[2].imshow(edge_map[0, 0], cmap="coolwarm")
axes[2].set_title("Feature map")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()


### Padding และ stride กำหนดขนาด feature map

- `padding` เติมขอบ เพื่อรักษาข้อมูลใกล้ขอบภาพ
- `stride` คือระยะที่ kernel ขยับในแต่ละครั้ง ค่ายิ่งมาก feature map ยิ่งเล็ก

สำหรับหนึ่งแกน ขนาดผลลัพธ์คือ `floor((input + 2×padding - kernel) / stride) + 1`


In [ ]:
for padding, stride in [(0, 1), (1, 1), (1, 2)]:
    result = F.conv2d(image_tensor, vertical_edge_kernel, padding=padding, stride=stride)
    print(f"padding={padding}, stride={stride} -> {tuple(result.shape[-2:])}")


### ตัวกรองแบบดั้งเดิมกับโปรเจกต์นี้

Week 4 ยังมี Gaussian filter, Sobel/Prewitt และ morphology เช่น erosion/dilation เครื่องมือเหล่านี้เหมาะกับ preprocessing ที่มีกฎชัดเจน

โปรเจกต์นี้ไม่ได้ใช้เป็นค่าเริ่มต้น เพราะอาจลบหรือขยายเส้นเล็ก ๆ จนตัวอักษรเปลี่ยนความหมาย และ CNN สามารถเรียน kernel จากข้อมูลเอง หากจะใช้ต้องทดลองเป็น ablation และวัดกับ validation ชุดเดิม


## 3. Activation และ pooling — Week 5 และ Week 7

Convolution อย่างเดียวเป็นการคำนวณเชิงเส้น การใส่ ReLU ทำให้เครือข่ายเรียนรูปแบบที่ซับซ้อนได้ โดยเปลี่ยนค่าติดลบเป็นศูนย์

Max pooling เลือกค่าสูงสุดในแต่ละบริเวณ ช่วยลดขนาด feature map และทำให้การขยับเล็กน้อยมีผลต่อ representation น้อยลง


In [ ]:
example_feature = torch.tensor([
    [[[-2.0, 1.0, 0.0, 3.0],
      [ 4.0,-1.0, 2.0, 1.0],
      [ 0.0, 2.0,-3.0, 1.0],
      [ 1.0, 0.0, 5.0, 2.0]]]
])

after_relu = F.relu(example_feature)
after_pool = F.max_pool2d(after_relu, kernel_size=2)

print("ก่อน ReLU:\n", example_feature[0, 0])
print("หลัง ReLU:\n", after_relu[0, 0])
print("หลัง 2×2 Max Pooling:\n", after_pool[0, 0])


### Batch Normalization และ Dropout

- Batch Normalization ปรับสเกล activation ภายใน batch ให้การฝึกเสถียรขึ้น
- Dropout สุ่มปิด activation บางส่วนเฉพาะตอน train เพื่อลดการพึ่ง feature จุดเดียวมากเกินไป

ทั้งสองชั้นทำงานต่างกันระหว่าง `model.train()` และ `model.eval()` ตัวเทรนจึงต้องสลับโหมดให้ถูก


In [ ]:
dropout = nn.Dropout(p=0.5)
values = torch.ones(10)

dropout.train()
print("Dropout ตอน train:", dropout(values))
dropout.eval()
print("Dropout ตอน eval: ", dropout(values))


## 4. CNN ที่ใช้เป็น baseline ในโปรเจกต์

Custom CNN ประกอบด้วย 4 บล็อก `Conv → BatchNorm → ReLU → MaxPool` แล้วใช้ Global Average Pooling, Dropout และ Linear classifier

Feature ชั้นต้นมักตอบสนองต่อขอบหรือเส้น ส่วนชั้นลึกค่อยรวมเป็นรูปร่างที่ซับซ้อนขึ้น

ใน Week 5 มีการสกัด feature แล้ววัด Manhattan/Euclidean distance ด้วย โปรเจกต์นี้ใช้ CNN เรียน feature และ classifier พร้อมกัน จึงไม่ได้ใช้ระยะทางเป็น baseline หากภายหลังทดลอง metric learning จึงค่อยนำแนวคิดระยะทางกลับมาใช้


In [ ]:
baseline_config = TrainConfig(
    architecture="custom_cnn",
    pretrained=False,
    image_size=32,
    batch_size=4,
    epochs=1,
    freeze_epochs=0,
    device="cpu",
)
baseline_model = build_model(baseline_config, classes=3)
print(baseline_model)

sample_batch = torch.randn(4, 3, 32, 32)
logits = baseline_model(sample_batch)
print("Input shape: ", tuple(sample_batch.shape))
print("Output shape:", tuple(logits.shape), "= 4 ภาพ × 3 คลาส")


## 5. Forward, loss, backpropagation และ optimizer — Week 6

หนึ่ง training step มีลำดับดังนี้:

```text
images → forward → logits → cross entropy loss → backward → optimizer step
```

`logits` ยังไม่ใช่ probability ส่วน Cross Entropy รับ logits ได้โดยตรง จึงไม่ใส่ softmax ก่อนคำนวณ loss


In [ ]:
targets = torch.tensor([0, 1, 2, 1])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=1e-3)

optimizer.zero_grad()
logits = baseline_model(sample_batch)
loss = criterion(logits, targets)
loss.backward()

first_parameter = next(baseline_model.parameters())
print("Loss:", loss.item())
print("Gradient norm:", first_parameter.grad.norm().item())

optimizer.step()
print("อัปเดต weight แล้ว 1 ครั้ง")


### Softmax ใช้ตอนแสดงผลทำนาย

Softmax แปลง logits ให้เป็น probability ที่รวมกันได้ 1 ส่วนคลาสที่มี probability สูงสุดคือคำตอบของโมเดล


In [ ]:
probabilities = logits.softmax(dim=1)
print("Logits ภาพแรก:", logits[0].detach())
print("Probabilities:", probabilities[0].detach())
print("ผลรวม:", probabilities[0].sum().item())
print("คลาสที่ทำนาย:", probabilities[0].argmax().item())


## 6. Transfer learning และ fine-tuning

โมเดล pretrained เคยเรียน feature ทั่วไปจากข้อมูลขนาดใหญ่ เราเปลี่ยน classifier ให้ตรงกับจำนวนคลาสของเรา แล้วฝึกสองช่วง:

1. freeze backbone และฝึก classifier ก่อน
2. unfreeze backbone แล้ว fine-tune ด้วย learning rate ที่ต่ำลง

ตัวอย่างนี้สร้าง ResNet-18 โดยไม่ดาวน์โหลด weights เพื่อแสดงโครงสร้างเท่านั้น การทดลองจริงใช้ `pretrained=True`


In [ ]:
transfer_config = TrainConfig(architecture="resnet18", pretrained=False)
transfer_model = build_model(transfer_config, classes=72)

for parameter in transfer_model.parameters():
    parameter.requires_grad_(False)
for parameter in transfer_model.get_classifier().parameters():
    parameter.requires_grad_(True)

all_parameters = sum(p.numel() for p in transfer_model.parameters())
trainable_parameters = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
print("Parameters ทั้งหมด:", f"{all_parameters:,}")
print("Parameters ที่ฝึกช่วงแรก:", f"{trainable_parameters:,}")


## 7. Data augmentation — Week 7

Augmentation สร้างมุมมองใหม่จากภาพจริงระหว่าง train เพื่อให้โมเดลทนต่อความต่างเล็กน้อย โปรเจกต์ใช้การหมุน ±5°, เลื่อน 3% และปรับขนาด 0.95–1.05

ไม่ใช้ horizontal/vertical flip เพราะอาจเปลี่ยนรูปและความหมายของตัวอักษรไทย Validation ไม่ทำ augmentation เพื่อให้การวัดผลคงที่


In [ ]:
pil_image = Image.fromarray((gray_image * 255).astype(np.uint8)).convert("RGB")
augmentation_config = TrainConfig(
    image_size=64,
    augmentation=True,
    pretrained=False,
    pad_value=255,
)
augment = make_transform(augmentation_config, training=True)

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for axis in axes:
    tensor = augment(pil_image)
    pixels = tensor.permute(1, 2, 0).numpy()
    pixels = pixels * np.array(augmentation_config.std) + np.array(augmentation_config.mean)
    axis.imshow(pixels.clip(0, 1))
    axis.axis("off")
plt.suptitle("Random augmentation samples")
plt.tight_layout()
plt.show()


## 8. ทำไมไม่ดู accuracy อย่างเดียว

ถ้าข้อมูลไม่สมดุล โมเดลอาจทายคลาสใหญ่เก่งและไม่เคยทายคลาสเล็ก Accuracy ยังดูสูงได้ ส่วน macro F1 คำนวณแต่ละคลาสแล้วเฉลี่ยโดยให้น้ำหนักเท่ากัน


In [ ]:
true_labels = [0] * 8 + [1] * 2
predicted_labels = [0] * 10

print("Accuracy:", accuracy_score(true_labels, predicted_labels))
print("Macro F1:", f1_score(true_labels, predicted_labels, average="macro"))

matrix = confusion_matrix(true_labels, predicted_labels, labels=[0, 1])
display(pd.DataFrame(matrix, index=["จริง: 0", "จริง: 1"], columns=["ทาย: 0", "ทาย: 1"]))


## 9. เชื่อมกับไฟล์จริงในโปรเจกต์

| แนวคิด | จุดที่ใช้จริง |
|---|---|
| RGB, resize, normalization | `make_transform()` ใน `src/train.py` |
| Augmentation | `RandomAffine` ใน `src/train.py` |
| Conv, BatchNorm, ReLU, Pooling | `CustomCNN` ใน `src/train.py` |
| Cross Entropy | `criterion` ใน `src/train.py` |
| Backpropagation | `loss.backward()` ใน training loop |
| AdamW และ LR scheduler | optimizer/scheduler ใน training loop |
| Freeze และ fine-tune | `freeze_epochs` ใน training loop |
| Softmax | evaluator และ `src/inference.py` |
| Macro F1 และ confusion matrix | evaluator ใน `src/train.py` |
| เปรียบเทียบโมเดลและ Optuna | `src/search.py` |

หลังเข้าใจ notebook นี้ ให้เปิด `01-model-search-lab.ipynb` เพื่อรันการทดลองจริง


## 10. คำถามเช็กความเข้าใจ

1. ทำไม Cross Entropy จึงรับ logits โดยไม่ต้อง softmax ก่อน?
2. padding มีผลต่อเส้นตัวอักษรที่อยู่ใกล้ขอบอย่างไร?
3. ทำไม augmentation ใช้เฉพาะ train?
4. หาก training accuracy 99% แต่ validation accuracy 75% น่าจะเกิดอะไรขึ้น?
5. ทำไม fine-tuning จึงใช้ learning rate ต่ำกว่าตอนฝึก classifier?

<details>
<summary>ดูแนวคำตอบ</summary>

1. Cross Entropy รวมการคำนวณ log-softmax ที่เสถียรกว่าไว้แล้ว
2. หากไม่ padding ข้อมูลบริเวณขอบจะถูกใช้ได้น้อยและ feature map จะเล็กลงเร็ว
3. Validation ต้องเป็นตัววัดคงที่และแทนภาพจริงที่ไม่เคยเห็น
4. โมเดลกำลัง overfit หรือ train/validation มี distribution ต่างกัน
5. เพื่อปรับ feature เดิมอย่างระมัดระวังและลดความเสี่ยงทำลาย pretrained weights

</details>
